# paper-translator: Colab サーバー

GPU ランタイムで実行する。API を起動して cloudflared で公開し、表示された URL をフロントの「API URL」欄に入れる。


In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/Kame09-M/paper-translator.git
%cd paper-translator/backend
!pip install -q -r requirements.txt

## モデルの準備

TODO: PLaMo 3 Translate の読み込み。transformers のバージョン・T4 での fp16 動作・`apply_chat_template` の `glossary` / `context` 引数の形式は、モデルカードで確認してから書く。


## API キー

Gemini の API キーは Colab の「シークレット」に `GEMINI_API_KEY` として登録し、ノートブックに直接書かない。


In [ ]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY") or ""
os.environ["MOCK"] = "1"  # 本番実装ができたら "0" にする

## API サーバー起動


In [ ]:
import subprocess

server = subprocess.Popen(
    ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=open("server.log", "w"), stderr=subprocess.STDOUT,
)
!sleep 3 && curl -s localhost:8000/api/health

## cloudflared で公開

出力に出る `https://*.trycloudflare.com` の URL をフロントに入れる。


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
!./cloudflared tunnel --url http://localhost:8000